# Social Media Emotion Analyzer - Model Development
This notebook shows the complete process for data loading, preprocessing, classical modeling, transformer fine-tuning, and evaluation.

## 1. Data Loading and Exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/emotion_clean.csv')
display(df.head())

In [ ]:
# Check class distribution
emotion_counts = df['emotion'].value_counts()
print(emotion_counts)

## 2. Text Preprocessing
Here we implement text cleaning directly without relying on external files.

In [ ]:
import re

STOPWORDS = {"a", "am", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "has", "have", "he", "i", "in", "is", "it", "its", "me", "my", "of", "on", "or", "our", "she", "so", "that", "the", "their", "them", "they", "this", "to", "was", "we", "were", "with", "you", "your"}

def clean_text(text: str) -> str:
    text = "" if text is None else str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)

sample = 'I am feeling so happy today!!! @friend #Joy'
print("Original:", sample)
print("Cleaned:", clean_text(sample))

# Apply to dataset
df['clean_text_notebook'] = df['text'].apply(clean_text)

## 3. Feature Extraction and Classical Models
We split the data and train TF-IDF Bigram + SVM, Count Bigram + Logistic Regression, and Count Bigram + Random Forest.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_recall_fscore_support

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text_notebook"],
    df["emotion"],
    test_size=0.2,
    random_state=42,
    stratify=df["emotion"]
)

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
model_results = []

def evaluate_and_plot(y_true, y_pred, model_name):
    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    model_results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": p,
        "Recall": r,
        "F1-Score": f1
    })
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=label_names)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


### TF-IDF Bigram + SVM

In [ ]:
pipeline_svm = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3))
])
pipeline_svm.fit(X_train, y_train)
y_pred_svm = pipeline_svm.predict(X_test)
print(classification_report(y_test, y_pred_svm))
evaluate_and_plot(y_test, y_pred_svm, "TF-IDF Bigram + SVM")


### Count Bigram + Logistic Regression

In [ ]:
pipeline_lr = Pipeline([
    ("count", CountVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
print(classification_report(y_test, y_pred_lr))
evaluate_and_plot(y_test, y_pred_lr, "Count Bigram + Logistic Regression")


### Count Bigram + Random Forest

In [ ]:
pipeline_rf = Pipeline([
    ("count", CountVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1))
])
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))
evaluate_and_plot(y_test, y_pred_rf, "Count Bigram + Random Forest")


## 4. DistilBERT Bonus Model
We use huggingface transformers to fine-tune DistilBERT.

In [ ]:
# Note: This requires 'transformers' and 'datasets' libraries.
import os
import numpy as np
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

label_to_id = {label: idx for idx, label in enumerate(label_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

# Sample a smaller subset for demonstration in notebook to speed up execution
df_sample = df.sample(n=1000, random_state=42).copy()
df_sample["label"] = df_sample["emotion"].map(label_to_id)

train_df, eval_df = train_test_split(df_sample[["text", "label"]], test_size=0.2, random_state=42, stratify=df_sample["label"])
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
)

training_args = TrainingArguments(
    output_dir="./distilbert_demo",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

# trainer.train() # Uncomment to run the training locally in the notebook
# preds = trainer.predict(eval_dataset)
# y_pred_distilbert = np.argmax(preds.predictions, axis=1)
# y_pred_labels = [id_to_label[p] for p in y_pred_distilbert]
# y_true_labels = [id_to_label[l] for l in eval_dataset['label']]
# evaluate_and_plot(y_true_labels, y_pred_labels, "DistilBERT (Demo Subset)")


## 5. Model Comparison Table
Here we compare the performance of all trained models based on Accuracy, Precision, Recall, and F1-Score.

In [ ]:
# Display comparison table
comparison_df = pd.DataFrame(model_results)
comparison_df = comparison_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
display(comparison_df)

# Plot comparison
plt.figure(figsize=(10, 5))
sns.barplot(x="F1-Score", y="Model", data=comparison_df, palette="viridis")
plt.title("Model Comparison by F1-Score")
plt.xlim(0, 1)
plt.show()
